# NatNorth Risk Intelligence Suite — ML Training Colab

Upload this notebook to Google Colab and run all cells.

**What it does**
1. Installs deps
2. Generates synthetic datasets (or loads CSVs if you upload them)
3. Trains all 6 models (2 per task)
4. Prints full metrics + saves JSON artifacts

**Design note:** narrate *why* each model exists (interpretability vs performance) while cells run.


In [ ]:
# Install dependencies
!pip -q install pandas numpy scikit-learn xgboost matplotlib


## 0 — Setup & helpers

In [ ]:

import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve, precision_recall_fscore_support
)
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')
RANDOM_STATE = 42
print('Ready')


## 1 — Payment Shield (APP fraud)

**Models:** Logistic Regression (coefficient explainability for regulated decisions) + XGBoost (non-linear interactions).

**Imbalance:** class_weight='balanced' for LR; scale_pos_weight = n_neg/n_pos for XGBoost.


In [ ]:

# If you uploaded payment_shield_dataset.csv to Colab, load it:
# df = pd.read_csv('payment_shield_dataset.csv')
# Otherwise generate here (same logic as ml/payment_shield.py)

def generate_payment(n=5000, fraud_rate=0.035, seed=42):
    rng = np.random.default_rng(seed)
    is_fraud = (rng.random(n) < fraud_rate).astype(int)
    fraud, legit = is_fraud == 1, is_fraud == 0
    n_f, n_l = int(fraud.sum()), int(legit.sum())
    amt = np.empty(n); newp = np.empty(n, int); hrs = np.empty(n)
    tod = np.empty(n, int); first = np.empty(n, int); z = np.empty(n)
    npay = np.empty(n, int); age = np.empty(n); intl = np.empty(n, int)
    amt[fraud] = rng.lognormal(6.2, 0.8, n_f); newp[fraud] = rng.binomial(1, 0.85, n_f)
    hrs[fraud] = rng.exponential(2.0, n_f)
    night = rng.random(n_f) < 0.72
    tod[fraud] = np.where(night, rng.choice([0,1,2,3,4,5,22,23], n_f), rng.integers(0,24,n_f))
    first[fraud] = rng.binomial(1, 0.80, n_f); z[fraud] = rng.normal(2.6, 0.75, n_f)
    npay[fraud] = rng.poisson(4.8, n_f); age[fraud] = rng.gamma(1.6, 80, n_f); intl[fraud] = rng.binomial(1, 0.40, n_f)
    amt[legit] = rng.lognormal(4.2, 0.85, n_l); newp[legit] = rng.binomial(1, 0.10, n_l)
    hrs[legit] = rng.exponential(18, n_l)
    day = rng.random(n_l) < 0.8
    tod[legit] = np.where(day, rng.integers(8,20,n_l), rng.integers(0,24,n_l))
    first[legit] = rng.binomial(1, 0.06, n_l); z[legit] = rng.normal(0, 0.8, n_l)
    npay[legit] = rng.poisson(1.0, n_l); age[legit] = rng.gamma(4.8, 210, n_l); intl[legit] = rng.binomial(1, 0.04, n_l)
    flip = rng.random(n) < 0.01
    is_fraud = np.where(flip, 1 - is_fraud, is_fraud)
    return pd.DataFrame({
        'transaction_amount': np.clip(amt,5,25000).round(2),
        'recipient_is_new_payee': newp, 'hours_since_last_login': np.clip(hrs,0.05,720).round(2),
        'time_of_day': tod, 'is_first_payment_to_recipient': first,
        'deviation_from_avg_transaction_zscore': z.round(3),
        'num_payments_today': np.clip(npay,0,25).astype(int),
        'account_age_days': np.clip(age,3,5000).round(1),
        'is_international': intl, 'is_fraud': is_fraud
    })

FEATURES = ['transaction_amount','recipient_is_new_payee','hours_since_last_login','time_of_day',
            'is_first_payment_to_recipient','deviation_from_avg_transaction_zscore',
            'num_payments_today','account_age_days','is_international']
df = generate_payment()
print(df.shape, 'fraud rate', df.is_fraud.mean())
df.head()


In [ ]:

X, y = df[FEATURES].values, df['is_fraud'].values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
sc = StandardScaler(); Xtrs, Xtes = sc.fit_transform(Xtr), sc.transform(Xte)
lr = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42).fit(Xtrs, ytr)
spw = (len(ytr) - ytr.sum()) / max(ytr.sum(), 1)
xgb = XGBClassifier(n_estimators=220, max_depth=5, learning_rate=0.07, scale_pos_weight=spw,
                    eval_metric='aucpr', random_state=42).fit(Xtr, ytr)
for name, prob in [('LR', lr.predict_proba(Xtes)[:,1]), ('XGB', xgb.predict_proba(Xte)[:,1])]:
    pred = (prob >= 0.5).astype(int)
    print(name, 'ROC-AUC', round(roc_auc_score(yte, prob),4),
          'PR-AUC', round(average_precision_score(yte, prob),4),
          'F1', round(f1_score(yte, pred),4))
print('LR coefficients (scaled space):')
for f, c in zip(FEATURES, lr.coef_[0]):
    print(f'  {f:40s} {c:+.4f}')


## 2 — Smart Categorizer

**Models:** TF-IDF + Logistic Regression (readable n-gram weights) + TF-IDF + Random Forest (noise-robust).


In [ ]:

# Prefer uploading categorizer_dataset.csv; else use a compact generator.
# For full templates, run ml/categorizer.py locally and upload the CSV.
from pathlib import Path
p = Path('categorizer_dataset.csv')
if p.exists():
    cat = pd.read_csv(p)
    print('Loaded CSV', cat.shape)
else:
    print('Upload categorizer_dataset.csv from public/datasets/ for full fidelity.')
    # minimal fallback
    rows = []
    for _ in range(800):
        rows.append({'description': 'TESCO STORES LONDON', 'category': 'Groceries'})
        rows.append({'description': 'UBER *TRIP', 'category': 'Transport'})
        rows.append({'description': 'NETFLIX.COM', 'category': 'Subscriptions'})
        rows.append({'description': 'SALARY ACME BACS', 'category': 'Salary/Income'})
    cat = pd.DataFrame(rows)
X, y = cat['description'].values, cat['category'].values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
vec = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=4000, sublinear_tf=True)
Xtr_t, Xte_t = vec.fit_transform(Xtr), vec.transform(Xte)
lr = LogisticRegression(max_iter=2000, C=2.0, random_state=42).fit(Xtr_t, ytr)
rf = RandomForestClassifier(n_estimators=200, max_depth=28, class_weight='balanced_subsample',
                            random_state=42, n_jobs=-1).fit(Xtr_t, ytr)
for name, model in [('LR', lr), ('RF', rf)]:
    pred = model.predict(Xte_t)
    print(name, 'Acc', round(accuracy_score(yte, pred),4), 'Macro-F1', round(f1_score(yte, pred, average='macro'),4))


## 3 — SME Pulse

**Models:** Logistic Regression (credit-committee audit trail) + Random Forest (interaction effects).


In [ ]:

p = Path('sme_pulse_dataset.csv')
if p.exists():
    panel = pd.read_csv(p)
else:
    print('Upload sme_pulse_dataset.csv for full panel; generating compact demo...')
    # Tiny demo panel — prefer the real CSV from the repo
    rng = np.random.default_rng(42)
    rows = []
    for sid in range(120):
        distressed = sid < 22
        for m in range(1,13):
            det = ((m-6)/6 if distressed and m>=7 else 0) * rng.uniform(0.5,1)
            inflow = rng.lognormal(10.2, 0.3) * (1-0.18*det)
            rows.append({
                'sme_id': f'SME-{sid:03d}', 'month': m,
                'monthly_inflow': inflow, 'monthly_outflow': inflow*(0.85+0.12*det),
                'overdraft_days_used': int(rng.poisson(1.5+4*det)),
                'late_supplier_payments': int(rng.poisson(0.6+2*det)),
                'inflow_volatility_3m': 0.1+0.2*det, 'cash_runway_months': max(0.3, 8*(1-0.35*det)),
                'distressed_within_6_months': int(distressed)
            })
    panel = pd.DataFrame(rows)

FEATS = ['monthly_inflow','monthly_outflow','overdraft_days_used','late_supplier_payments','inflow_volatility_3m','cash_runway_months']
snap = panel[panel.month == panel.month.max()].copy()
X, y = snap[FEATS].values, snap['distressed_within_6_months'].values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
sc = StandardScaler(); Xtrs, Xtes = sc.fit_transform(Xtr), sc.transform(Xte)
lr = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42).fit(Xtrs, ytr)
rf = RandomForestClassifier(n_estimators=250, max_depth=8, class_weight='balanced_subsample',
                            random_state=42, n_jobs=-1).fit(Xtr, ytr)
for name, prob in [('LR', lr.predict_proba(Xtes)[:,1]), ('RF', rf.predict_proba(Xte)[:,1])]:
    print(name, 'ROC-AUC', round(roc_auc_score(yte, prob),4), 'PR-AUC', round(average_precision_score(yte, prob),4))


## Model design notes

1. **Why keep Logistic Regression when XGBoost/RF can win?** Regulated banking decisions need coefficient-level explainability and challengeable signs.
2. **Why PR-AUC for fraud?** With ~3–4% positives, ROC can look strong while precision on the positive class is weak.
3. **Why stratified splits?** Preserve rare positive rates in both train and test.
4. **Why class_weight / scale_pos_weight?** So the minority class is not ignored by accuracy-driven training.
5. **Why trajectories for SME?** Account-level labels + month-level scoring shows *lead time* before distress.
